In [ ]:
# imports and configuration

from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os
import sys

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

if device != "cuda":
    raise RuntimeError("BitsAndBytes Linear8bitLt generally requires CUDA for this experiment.")

project_root = Path("/home/lecacca/VSCode Environment/BERT-Testing")
helper_dir = project_root / "scripts" / "quantize_sublayer"

if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

import quantize_experiments as manual_helpers
import quantize_experiments_bitsandbytes as bnb_helpers

model_name = "/home/lecacca/VSCode Environment/BERT-Testing/bert_uncased_L-2_H-128_A-2"

texts_path = Path(
    "/home/lecacca/VSCode Environment/BERT-Testing/visualization/scatter_plot_bitsandbytes/masked_texts.txt"
)

default_out_dir = Path(
    "/home/lecacca/VSCode Environment/BERT-Testing/visualization/scatter_plot_bitsandbytes/results"
)

# If running from sbatch, use RESULTS_DIR. Otherwise use default_out_dir.
out_dir = Path(os.environ.get("RESULTS_DIR", default_out_dir))
out_dir.mkdir(parents=True, exist_ok=True)

print("Output directory:", out_dir)

block_index = 0

sublayers = [
    "attention.self.query",
    "attention.self.key",
    "attention.self.value",
    "attention.output.dense",
    "intermediate.dense",
    "output.dense",
]

In [ ]:
# load masked texts

def load_masked_texts(path):
    texts = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            if line.startswith("#"):
                continue

            texts.append(line)

    return texts


masked_texts = load_masked_texts(texts_path)

print(f"Loaded {len(masked_texts)} masked texts")
for i, text in enumerate(masked_texts[:5]):
    print(f"{i}: {text}")

In [ ]:
# notebook-specific inference helpers

def move_inputs_to_model_device(inputs, model):
    device = next(model.parameters()).device
    return {k: v.to(device) for k, v in inputs.items()}


def get_mask_logits(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt")
    inputs = move_inputs_to_model_device(inputs, model)

    mask_positions = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=False)

    if mask_positions.numel() == 0:
        raise ValueError(f"No [MASK] token found in text: {text}")

    if mask_positions.shape[0] > 1:
        print(f"Warning: multiple [MASK] tokens found. Using first one for text: {text}")

    mask_index = mask_positions[0, 1].item()

    with torch.inference_mode():
        outputs = model(**inputs)

    logits = outputs.logits
    mask_logits = logits[0, mask_index, :].detach().float().cpu()

    return mask_logits


def get_top1_token_from_logits(logits, tokenizer):
    top_idx = torch.argmax(logits).item()
    return tokenizer.convert_ids_to_tokens([top_idx])[0]


def get_topk_tokens_from_logits(logits, tokenizer, k=5):
    top_values, top_indices = torch.topk(logits, k)

    tokens = tokenizer.convert_ids_to_tokens(
        top_indices.detach().cpu().tolist()
    )

    scores = top_values.detach().cpu().tolist()

    return tokens, scores

In [ ]:
# wrappers around imported helper functions

def get_module_by_name(root, module_path):
    return manual_helpers.get_module_by_name(root, module_path)


def quantize_one_sublayer_manual(model, block_index, sub_layer, verbose=False):
    """
    Uses quantize_linear_weights_per_channel from:

    scripts/quantize_sublayer/quantize_experiments.py
    """

    layer = get_module_by_name(
        model.bert.encoder.layer[block_index],
        sub_layer,
    )

    info = manual_helpers.quantize_linear_weights_per_channel(
        layer,
        num_bits=8,
        verbose=verbose,
    )

    return info


def quantize_one_sublayer_bnb_8bit(model, block_index, sub_layer):
    """
    Uses replace_one_linear_with_bnb_8bit from:

    scripts/quantize_sublayer/quantize_experiments_bitsandbytes.py

    This function modifies the model in-place.
    """

    bnb_helpers.replace_one_linear_with_bnb_8bit(
        model.bert.encoder.layer[block_index],
        sub_layer,
    )

    layer = get_module_by_name(
        model.bert.encoder.layer[block_index],
        sub_layer,
    )

    return layer

In [ ]:
# precompute FP32 baseline logits

tokenizer = AutoTokenizer.from_pretrained(model_name)

baseline_model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)
baseline_model.eval()

baseline_logits_by_text = {}
baseline_top1_by_text = {}

for text_index, text in enumerate(masked_texts):
    logits = get_mask_logits(baseline_model, tokenizer, text)

    baseline_logits_by_text[text_index] = logits
    baseline_top1_by_text[text_index] = get_top1_token_from_logits(logits, tokenizer)

print("Finished baseline inference.")
print("Example baseline top-1 predictions:")

for i in range(min(5, len(masked_texts))):
    print(i, masked_texts[i], "->", baseline_top1_by_text[i])

In [ ]:
# run manual and BitsAndBytes quantization experiments

results = []

for sub_layer in sublayers:
    print(f"\n=== Sublayer: block {block_index}.{sub_layer} ===")

    # ---------------------------------------------------------------------
    # Fresh model for manual per-channel fake quantization
    # ---------------------------------------------------------------------

    manual_model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)
    manual_model.eval()

    manual_quant_info = quantize_one_sublayer_manual(
        manual_model,
        block_index=block_index,
        sub_layer=sub_layer,
        verbose=True,
    )

    # ---------------------------------------------------------------------
    # Fresh model for BitsAndBytes quantization
    # ---------------------------------------------------------------------

    # Important:
    # Replace the layer before moving model to CUDA.
    bnb_model = AutoModelForMaskedLM.from_pretrained(model_name)
    bnb_model.eval()

    bnb_layer = quantize_one_sublayer_bnb_8bit(
        bnb_model,
        block_index=block_index,
        sub_layer=sub_layer,
    )

    # Moving to CUDA initializes the BitsAndBytes quantized layer behavior.
    bnb_model.to(device)
    bnb_model.eval()

    bnb_layer = get_module_by_name(
        bnb_model.bert.encoder.layer[block_index],
        sub_layer,
    )

    # ---------------------------------------------------------------------
    # Evaluate all masked texts
    # ---------------------------------------------------------------------

    for text_index, text in enumerate(masked_texts):
        baseline_logits = baseline_logits_by_text[text_index]

        manual_logits = get_mask_logits(manual_model, tokenizer, text)
        bnb_logits = get_mask_logits(bnb_model, tokenizer, text)

        manual_abs_diff = (manual_logits - baseline_logits).abs()
        bnb_abs_diff = (bnb_logits - baseline_logits).abs()

        manual_mean_abs_diff = manual_abs_diff.mean().item()
        manual_max_abs_diff = manual_abs_diff.max().item()

        bnb_mean_abs_diff = bnb_abs_diff.mean().item()
        bnb_max_abs_diff = bnb_abs_diff.max().item()

        manual_top1 = get_top1_token_from_logits(manual_logits, tokenizer)
        bnb_top1 = get_top1_token_from_logits(bnb_logits, tokenizer)
        baseline_top1 = baseline_top1_by_text[text_index]

        manual_top5, manual_top5_scores = get_topk_tokens_from_logits(
            manual_logits,
            tokenizer,
            k=5,
        )

        bnb_top5, bnb_top5_scores = get_topk_tokens_from_logits(
            bnb_logits,
            tokenizer,
            k=5,
        )

        baseline_top5, baseline_top5_scores = get_topk_tokens_from_logits(
            baseline_logits,
            tokenizer,
            k=5,
        )

        manual_top5_overlap = len(set(baseline_top5) & set(manual_top5))
        bnb_top5_overlap = len(set(baseline_top5) & set(bnb_top5))

        manual_top5_rank_matches = sum(
            b_tok == q_tok
            for b_tok, q_tok in zip(baseline_top5, manual_top5)
        )

        bnb_top5_rank_matches = sum(
            b_tok == q_tok
            for b_tok, q_tok in zip(baseline_top5, bnb_top5)
        )

        results.append({
            "block_index": block_index,
            "sub_layer": sub_layer,
            "text_index": text_index,
            "text": text,

            "baseline_top1": baseline_top1,
            "manual_top1": manual_top1,
            "bnb_top1": bnb_top1,

            "baseline_top5": baseline_top5,
            "manual_top5": manual_top5,
            "bnb_top5": bnb_top5,

            "manual_top1_match": manual_top1 == baseline_top1,
            "bnb_top1_match": bnb_top1 == baseline_top1,

            "manual_top5_overlap": manual_top5_overlap,
            "bnb_top5_overlap": bnb_top5_overlap,

            "manual_top5_rank_matches": manual_top5_rank_matches,
            "bnb_top5_rank_matches": bnb_top5_rank_matches,

            "manual_mean_abs_logit_diff": manual_mean_abs_diff,
            "bnb_mean_abs_logit_diff": bnb_mean_abs_diff,

            "manual_max_abs_logit_diff": manual_max_abs_diff,
            "bnb_max_abs_logit_diff": bnb_max_abs_diff,
        })

    del manual_model
    del bnb_model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df = pd.DataFrame(results)

print("Finished experiments.")
display(df.head())

out_csv = out_dir / "manual_vs_bitsandbytes_results.csv"
df.to_csv(out_csv, index=False)

print("Saved:", out_csv)

In [ ]:
# scatter plot mean absolute logit difference

plt.figure(figsize=(9, 7))

sns.scatterplot(
    data=df,
    x="manual_mean_abs_logit_diff",
    y="bnb_mean_abs_logit_diff",
    hue="sub_layer",
    style="sub_layer",
    s=80,
    alpha=0.85,
)

max_val = max(
    df["manual_mean_abs_logit_diff"].max(),
    df["bnb_mean_abs_logit_diff"].max(),
)

plt.plot([0, max_val], [0, max_val], "k--", linewidth=1, label="y = x")

plt.xlabel("Manual per-channel fake INT8 mean absolute logit diff")
plt.ylabel("BitsAndBytes Linear8bitLt mean absolute logit diff")

plt.title(
    f"Manual vs BitsAndBytes INT8 Quantization\n"
    f"Block {block_index}, mean absolute [MASK] logit drift"
)

plt.grid(True, alpha=0.3)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

fig_path = out_dir / "scatter_mean_abs_logit_diff_manual_vs_bitsandbytes.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")

print("Saved:", fig_path)

plt.show()

In [ ]:
# scatter plot max absolute logit difference

plt.figure(figsize=(9, 7))

sns.scatterplot(
    data=df,
    x="manual_max_abs_logit_diff",
    y="bnb_max_abs_logit_diff",
    hue="sub_layer",
    style="sub_layer",
    s=80,
    alpha=0.85,
)

max_val = max(
    df["manual_max_abs_logit_diff"].max(),
    df["bnb_max_abs_logit_diff"].max(),
)

plt.plot([0, max_val], [0, max_val], "k--", linewidth=1, label="y = x")

plt.xlabel("Manual per-channel fake INT8 max absolute logit diff")
plt.ylabel("BitsAndBytes Linear8bitLt max absolute logit diff")

plt.title(
    f"Manual vs BitsAndBytes INT8 Quantization\n"
    f"Block {block_index}, max absolute [MASK] logit drift"
)

plt.grid(True, alpha=0.3)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

fig_path = out_dir / "scatter_max_abs_logit_diff_manual_vs_bitsandbytes.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")

print("Saved:", fig_path)

plt.show()

In [ ]:
# scatter plot by sublayer

g = sns.FacetGrid(
    df,
    col="sub_layer",
    col_wrap=3,
    height=4,
    sharex=False,
    sharey=False,
)

g.map_dataframe(
    sns.scatterplot,
    x="manual_mean_abs_logit_diff",
    y="bnb_mean_abs_logit_diff",
    s=70,
    alpha=0.85,
)

for ax in g.axes.flatten():
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    max_val = max(xlim[1], ylim[1])

    ax.plot([0, max_val], [0, max_val], "k--", linewidth=1)
    ax.grid(True, alpha=0.3)

g.set_axis_labels(
    "Manual per-channel fake INT8 mean abs logit diff",
    "BitsAndBytes Linear8bitLt mean abs logit diff",
)

g.fig.suptitle(
    f"Manual vs BitsAndBytes INT8 Quantization by Sublayer, Block {block_index}",
    y=1.03,
)

plt.tight_layout()

fig_path = out_dir / "facet_mean_abs_logit_diff_manual_vs_bitsandbytes.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")

print("Saved:", fig_path)

plt.show()